# 1. Dataset

In [2]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# -------------------- Dataset --------------------

# X shape = (10,2)

X = np.array([
    [2,3],
    [3,4],
    [4,5],
    [5,6],
    [6,5],
    [7,8],
    [8,7],
    [9,10],
    [10,9],
    [11,12]
], dtype=float)

# y shape = (10,)

y = np.array([0,0,0,0,1,1,1,1,1,1])

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (10, 2)
y Shape: (10,)


# 2. Sklearn

In [3]:
sk_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=2,
    random_state=42
)

sk_model.fit(X,y)

sk_pred = sk_model.predict(X)

print(sk_pred)

print("Accuracy:",sk_model.score(X,y))

[0 0 0 0 1 1 1 1 1 1]
Accuracy: 1.0


# 3. From Scratch

In [4]:
import numpy as np

class Node:

    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, value=None):

        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTreeScratch:

    def __init__(self, max_depth=2):
        self.max_depth = max_depth

    def gini(self, y):

        classes = np.unique(y)

        impurity = 1

        for c in classes:

            p = np.sum(y==c)/len(y)

            impurity -= p**2

        return impurity


    def best_split(self,X,y):

        best_gini = 999

        best_feature = None
        best_threshold = None

        n_samples,n_features = X.shape

        for feature in range(n_features):

            thresholds=np.unique(X[:,feature])

            for threshold in thresholds:

                left = X[:,feature] <= threshold
                right = X[:,feature] > threshold

                if np.sum(left)==0 or np.sum(right)==0:
                    continue

                g_left = self.gini(y[left])
                g_right = self.gini(y[right])

                weighted = (
                    np.sum(left)/n_samples * g_left
                    +
                    np.sum(right)/n_samples * g_right
                )

                if weighted < best_gini:

                    best_gini = weighted
                    best_feature = feature
                    best_threshold = threshold

        return best_feature,best_threshold


    def build(self,X,y,depth):

        classes=np.unique(y)

        if len(classes)==1:
            return Node(value=classes[0])

        if depth==self.max_depth:
            return Node(value=np.bincount(y).argmax())

        feature,threshold=self.best_split(X,y)

        if feature is None:
            return Node(value=np.bincount(y).argmax())

        left = X[:,feature] <= threshold
        right = X[:,feature] > threshold

        left_node=self.build(X[left],y[left],depth+1)
        right_node=self.build(X[right],y[right],depth+1)

        return Node(
            feature,
            threshold,
            left_node,
            right_node
        )

    def fit(self,X,y):

        self.root=self.build(X,y,0)

    def traverse(self,x,node):

        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self.traverse(x,node.left)

        return self.traverse(x,node.right)

    def predict(self,X):

        return np.array([
            self.traverse(x,self.root)
            for x in X
        ])


my_model=DecisionTreeScratch(max_depth=2)

my_model.fit(X,y)

my_pred=my_model.predict(X)

print(my_pred)

[0 0 0 0 1 1 1 1 1 1]


# 4. Accuracy

In [6]:
from sklearn.metrics import accuracy_score

print("\n===== Sklearn =====")
print(accuracy_score(y,sk_pred))

print("\n===== Scratch =====")
print(accuracy_score(y,my_pred))


===== Sklearn =====
1.0

===== Scratch =====
1.0


# 5. Decision Tree (Gini)

## Step 1 : Calculate Gini Impurity

$$
Gini
=
1-
\sum_{i=1}^{C}
p_i^2
$$

where

- $p_i$ = probability of class $i$

---

## Step 2 : Try Every Feature

For every feature,

try every possible threshold.

Example

Feature 1

$$
2,3,4,5,6...
$$

---

## Step 3 : Split Dataset

Left

$$
X \le threshold
$$

Right

$$
X > threshold
$$

---

## Step 4 : Calculate Weighted Gini

$$
G
=
\frac{N_L}{N}G_L
+
\frac{N_R}{N}G_R
$$

where

- $N_L$ = Left samples

- $N_R$ = Right samples

---

## Step 5 : Choose Best Split

Choose

$$
\boxed{
\text{Minimum Weighted Gini}
}
$$

---

## Step 6 : Repeat Recursively

Repeat until

- Pure node

or

- Max depth reached

or

- No split possible

---

## Step 7 : Prediction

Start from root.

If

$$
Feature
\le
Threshold
$$

Go Left

Else

Go Right

Repeat until leaf.

Return leaf class.

---

# Shapes

| Variable | Shape |
|-----------|-------|
| $X$ | $(n_{samples},n_{features})$ |
| $y$ | $(n_{samples},)$ |
| Left Node | Variable |
| Right Node | Variable |
| Prediction | Scalar |

# 6. Entropy

In [8]:
import numpy as np

class Node:

    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, value=None):

        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTreeEntropy:

    def __init__(self, max_depth=2):
        self.max_depth = max_depth


    # ---------------- Entropy ----------------

    def entropy(self, y):

        classes = np.unique(y)

        entropy = 0

        for c in classes:

            p = np.sum(y == c) / len(y)

            entropy -= p * np.log2(p)

        return entropy


    # ---------------- Information Gain ----------------

    def information_gain(self, parent, left, right):

        n = len(parent)

        parent_entropy = self.entropy(parent)

        left_entropy = self.entropy(left)

        right_entropy = self.entropy(right)

        weighted_entropy = (
            len(left) / n * left_entropy
            +
            len(right) / n * right_entropy
        )

        return parent_entropy - weighted_entropy


    # ---------------- Best Split ----------------

    def best_split(self, X, y):

        best_gain = -1

        best_feature = None
        best_threshold = None

        n_samples, n_features = X.shape

        for feature in range(n_features):

            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:

                left = X[:, feature] <= threshold
                right = X[:, feature] > threshold

                if np.sum(left) == 0 or np.sum(right) == 0:
                    continue

                gain = self.information_gain(
                    y,
                    y[left],
                    y[right]
                )

                if gain > best_gain:

                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold


    # ---------------- Build Tree ----------------

    def build(self, X, y, depth):

        classes = np.unique(y)

        if len(classes) == 1:
            return Node(value=classes[0])

        if depth == self.max_depth:
            return Node(value=np.bincount(y).argmax())

        feature, threshold = self.best_split(X, y)

        if feature is None:
            return Node(value=np.bincount(y).argmax())

        left = X[:, feature] <= threshold
        right = X[:, feature] > threshold

        left_node = self.build(X[left], y[left], depth + 1)
        right_node = self.build(X[right], y[right], depth + 1)

        return Node(
            feature,
            threshold,
            left_node,
            right_node
        )


    # ---------------- Fit ----------------

    def fit(self, X, y):

        self.root = self.build(X, y, 0)


    # ---------------- Prediction ----------------

    def traverse(self, x, node):

        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self.traverse(x, node.left)

        return self.traverse(x, node.right)


    def predict(self, X):

        return np.array([
            self.traverse(x, self.root)
            for x in X
        ])


# ---------------- Test ----------------

my_model = DecisionTreeEntropy(max_depth=2)

my_model.fit(X, y)

my_pred = my_model.predict(X)

print(my_pred)

[0 0 0 0 1 1 1 1 1 1]


# 7. Entropy

# Decision Tree (Entropy)

## Step 1: Calculate Entropy

Entropy measures the impurity of a node.

$$
Entropy
=
-\sum_{i=1}^{C}
p_i\log_2(p_i)
$$

where

- $p_i$ = Probability of class $i$
- $C$ = Number of classes

Code

```python
entropy = 0

for c in classes:

    p = np.sum(y == c) / len(y)

    entropy -= p * np.log2(p)
```

---

## Step 2: Try Every Feature

For every feature,

try every possible threshold.

Example

Feature 1

$$
2,3,4,5,6,\ldots
$$

---

## Step 3: Split Dataset

Left

$$
X \le \text{Threshold}
$$

Right

$$
X > \text{Threshold}
$$

---

## Step 4: Calculate Parent Entropy

$$
Entropy_{parent}
=
Entropy(y)
$$

Code

```python
parent_entropy = self.entropy(parent)
```

---

## Step 5: Calculate Left and Right Entropy

$$
Entropy_{left}
=
Entropy(y_{left})
$$

$$
Entropy_{right}
=
Entropy(y_{right})
$$

Code

```python
left_entropy = self.entropy(left)
right_entropy = self.entropy(right)
```

---

## Step 6: Calculate Weighted Entropy

$$
WeightedEntropy
=
\frac{N_L}{N}
Entropy_{left}
+
\frac{N_R}{N}
Entropy_{right}
$$

where

- $N_L$ = Left samples
- $N_R$ = Right samples
- $N$ = Total samples

Code

```python
weighted_entropy = (
    len(left)/n * left_entropy
    +
    len(right)/n * right_entropy
)
```

---

## Step 7: Calculate Information Gain

$$
InformationGain
=
Entropy_{parent}
-
WeightedEntropy
$$

Code

```python
gain = parent_entropy - weighted_entropy
```

---

## Step 8: Choose Best Split

Choose the split having

$$
\boxed{
\text{Maximum Information Gain}
}
$$

Code

```python
if gain > best_gain:
```

---

## Step 9: Repeat Recursively

Repeat until

- Node becomes pure

or

- Maximum depth reached

or

- No valid split exists

---

## Step 10: Prediction

Start from the root node.

If

$$
Feature
\le
Threshold
$$

Go Left

Else

Go Right

Repeat until a leaf node is reached.

Return the leaf class.

---

# Final Formula

$$
Entropy
=
-\sum p_i\log_2(p_i)
$$

$$
WeightedEntropy
=
\frac{N_L}{N}Entropy_{left}
+
\frac{N_R}{N}Entropy_{right}
$$

$$
\boxed{
InformationGain
=
Entropy_{parent}
-
WeightedEntropy
}
$$

Choose the split with the **highest Information Gain**.

---

# Shapes

| Variable | Shape |
|-----------|-------|
| $X$ | $(n_{samples},n_{features})$ |
| $y$ | $(n_{samples},)$ |
| Parent Labels | $(n_{samples},)$ |
| Left Labels | Variable |
| Right Labels | Variable |
| Information Gain | Scalar |
| Prediction | Scalar |